In [1]:
import base64
import io
import json
import os
import re
import stringcase
import voxeloo
import numpy as np
from dataclasses import dataclass
from glob import glob
from string import Template
from typing import Dict
from PIL import Image

In [2]:
def load_texture(path):
    return Image.open(f"../../data/{path}")

In [3]:
def flip_texture(img):
    return img.transpose(Image.FLIP_TOP_BOTTOM)

In [4]:
def encode_texture(image):
    with io.BytesIO() as output:
        image.save(output, format="PNG")
        return base64.b64encode(output.getvalue()).decode("utf-8")

In [5]:
@dataclass
class Sample:
    side: str
    top: str
    bottom: str
    mrea: str
    mreaside: str
    mreatop: str
    mreabottom: str

@dataclass
class Block:
    black: Dict[str, Sample]
    white: Dict[str, Sample]

In [6]:
CUBE_TEXTURE_TEMPLATE = Template("""{
  "x_neg": "$side",
  "x_pos": "$side",
  "y_neg": "$bottom",
  "y_pos": "$top",
  "z_neg": "$side",
  "z_pos": "$side"
}""")

SAMPLE_TEMPLATE = Template("""{
  "color": $color,
  "mrea": $mrea
}"""
)

BLOCK_TEMPLATE = Template("""{
  "name": "$name",
  "label": "$label",
  "index": 1,
  "white": [
    $white
  ],
  "black": [
    $black
  ]
}"""
)

def json_format(s: str):
    return json.dumps(json.loads(s), indent=2)

def get_mrea(sample, side):
  potentialTextures = ['mrea'+side, 'mrea']
  for attr in potentialTextures:
    if hasattr(sample, attr):
      potential = getattr(sample, attr)
      if potential != "":
        return potential
  return "blocks/textures/shared/default_mrea.png"

def block_to_json(name: str, block: Block):
    black = [
        SAMPLE_TEMPLATE.substitute({
            "color": CUBE_TEXTURE_TEMPLATE.substitute({
              "side": encode_texture(flip_texture(load_texture(sample.side))),
              "top": encode_texture(flip_texture(load_texture(sample.top))),
              "bottom": encode_texture(flip_texture(load_texture(sample.bottom))),
            }),
            "mrea": CUBE_TEXTURE_TEMPLATE.substitute({
              "side": encode_texture(flip_texture(load_texture(get_mrea(sample, "side")))),
              "bottom": encode_texture(flip_texture(load_texture(get_mrea(sample, "bottom")))),
              "top": encode_texture(flip_texture(load_texture(get_mrea(sample, "top"))))
            })
        })
        for sample in block.black.values()
    ]
    white = [
        SAMPLE_TEMPLATE.substitute({
            "color": CUBE_TEXTURE_TEMPLATE.substitute({
              "side": encode_texture(flip_texture(load_texture(sample.side))),
              "top": encode_texture(flip_texture(load_texture(sample.top))),
              "bottom": encode_texture(flip_texture(load_texture(sample.bottom))),
            }),
            "mrea": CUBE_TEXTURE_TEMPLATE.substitute({
              "side": encode_texture(flip_texture(load_texture(get_mrea(sample, "side")))),
              "bottom": encode_texture(flip_texture(load_texture(get_mrea(sample, "bottom")))),
              "top": encode_texture(flip_texture(load_texture(get_mrea(sample, "top"))))
            })
        })
        for sample in block.white.values()
    ]
    
    return json_format(
        BLOCK_TEMPLATE.substitute({
            "name": stringcase.titlecase(name),
            "label": "1",
            "black": ",".join(black),
            "white": ",".join(white),
        })
    )

In [7]:
blocks = {}
for path in glob("../../data/blocks/textures/*.png"):
    filename = os.path.basename(path)
    name, tile, variant, side = re.match("(.+)_([^_]+)_([^_]+)_([^_]+).png", filename).groups()
    
    
    block = blocks.setdefault(name, Block({}, {}))
    sample = getattr(block, tile).setdefault(variant, Sample("", "", "", "", "", "", ""))
    setattr(sample, side, f"blocks/textures/{filename}")

In [8]:
for name, block in blocks.items():
    output = block_to_json(name, block)
    with open(f"../../data/blocks/{name}.json", "w") as f: 
        f.write(output)

### Below is some hacky code to flip textures

In [ ]:
def get_textures(path):
    name = os.path.basename(path).split(".")[0]
    
    with open(path) as f:
        block = json.loads(f.read())
    
    textures = {}
    for tile in ["black", "white"]:
        for i, sample in enumerate(block[tile]):
            textures[f"{name}_{tile}_v{i}_side.png"] = sample["color"]["x_neg"]
            textures[f"{name}_{tile}_v{i}_top.png"] = sample["color"]["y_pos"]
            textures[f"{name}_{tile}_v{i}_bottom.png"] = sample["color"]["y_neg"]
    return textures

In [ ]:
def load_image_from_base64(data):
    return Image.open(io.BytesIO(base64.b64decode(data)))

In [ ]:
block_files = glob("../../data/blocks/*.json")

In [ ]:
for path in block_files:
    textures = get_textures(path)
    for name, data in textures.items():
        img = load_image_from_base64(data)
        img.transpose(Image.FLIP_TOP_BOTTOM).save(f"../../data/exports/textures/blocks/{name}")

### Hack for generating an emissive texture

In [ ]:
template = load_texture("blocks/textures/shared/default_mrea.png")

In [ ]:
mrea = np.array(template)

In [ ]:
mrea[:, :, 1] = 200
mrea[:, :, 2] = 255

In [ ]:
Image.fromarray(mrea).save("../../data/blocks/textures/led_black_v0_mrea.png")